# 🥊 Classification Showdown: The Masterclass
Welcome to your final test of classical Machine Learning algorithms!
You know the math and the theory. Now it's time to see **Decision Trees**, **Support Vector Machines (SVM)**, **K-Nearest Neighbors (KNN)**, and **Gaussian Naive Bayes** battle it out on a real biological dataset.

## The Objective
We are going to use the **Breast Cancer Wisconsin (Diagnostic)** dataset. 
It contains 30 numerical features computed from a digitized image of a fine needle aspirate (FNA) of a breast mass. 
The features describe characteristics of the cell nuclei (radius, texture, perimeter, area, etc.).

**Our Goal**: Predict whether a tumor is `Malignant` (Cancerous) or `Benign` (Healthy).

## 1. Loading the Biological Data & Preprocessing

First, we load the dataset and do the MOST IMPORTANT step for algorithms like SVM and KNN: **Feature Scaling**.

### The Scaling Trap
- **SVM & KNN** rely heavily on distances between data points (KNN literally measures Euclidean distance, SVM maximizes geometric distance of the margin). If one feature is measured in thousands (like Area) and another in fractions (like Smoothness), Area will completely dominate the distance calculation!
- **Decision Trees & Naive Bayes**, however, DO NOT CARE about scaling! Trees just split on thresholds, and Naive Bayes looks at probabilistic distributions independently.

Let's scale the data so all algorithms fight fairly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

# Load Breast Cancer Data
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target # 0 = Malignant, 1 = Benign

# Split data (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Scaling (Crucial for SVM & KNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Dataset shape: {X.shape}")
print(f"Training features (scaled):\n Mean: {np.round(np.mean(X_train_scaled[:, 0]), 2)}, Variance: {np.round(np.var(X_train_scaled[:, 0]), 2)}")


## 2. Algorithm 1: Decision Tree 🌳
The Decision Tree builds a flowchart. It looks at the features and asks questions like, *"Is the tumor radius > x? If yes, it's likely malignant."*

**Pros:** Extremely interpretable, doesn't need scaled data.
**Cons:** Prone to overfitting (memorizing the training data).

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and train
# Note: We can use the regular X_train here, it doesn't need X_train_scaled!
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)

# Predict
y_pred_dt = dt_model.predict(X_test)

print("Decision Tree Results:")
print("-" * 20)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")


## 3. Algorithm 2: Support Vector Machine (SVM) ⚔️
SVM tries to draw a "hyperplane" (a line/plane) that separates Malignant from Benign, but it wants to maximize the *margin* (the gap) between the two classes.

Since biology is rarely a straight line, we use the **'rbf' (Radial Basis Function) Kernel**, which mathematically projects the data into an infinite number of dimensions to easily separate the non-linear classes.

In [ ]:
from sklearn.svm import SVC

# Initialize and train
# MUST use scaled data!
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)

# Predict
y_pred_svm = svm_model.predict(X_test_scaled)

print("Support Vector Machine (SVM) Results:")
print("-" * 37)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_svm):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_svm):.4f}")


## 4. Algorithm 3: K-Nearest Neighbors (KNN) 📍
KNN is a "lazy" learner. It doesn't really "train" a model equations. Instead, to predict a new tumor, it simply looks at the `K` closest tumors in the training data and takes a majority vote!

If K=5, and 4 neighbors are Malignant and 1 is Benign, it predicts Malignant.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Initialize and train
# MUST use scaled data!
knn_model = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2) # p=2 is Euclidean distance
knn_model.fit(X_train_scaled, y_train)

# Predict
y_pred_knn = knn_model.predict(X_test_scaled)

print("K-Nearest Neighbors (KNN) Results:")
print("-" * 34)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_knn):.4f}")


## 5. Algorithm 4: Gaussian Naive Bayes 📊
Naive Bayes applies Bayes' Theorem of probability. The word "Naive" means it assumes every feature (radius, texture, etc.) is completely independent of the others.
"Gaussian" means it assumes each feature's values form a Bell Curve (Normal Distribution).

It calculates probabilities for each class and picks the highest one!

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Initialize and train
# Note: Naive Bayes handles distances via probabilities, so scaling isn't strictly necessary, 
# although it doesn't usually hurt. We'll use the original unscaled data to show it still works!
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict
y_pred_nb = nb_model.predict(X_test)

print("Gaussian Naive Bayes Results:")
print("-" * 29)
print(f"Accuracy:  {accuracy_score(y_test, y_pred_nb):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_nb):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_nb):.4f}")


## 6. 🏆 THE FINAL SHOWDOWN
Let's put all their predictions side-by-side and see who won! In medical contexts like detecting cancer, **Recall** is often the most critical metric. High recall means we want to minimize False Negatives (telling a patient they are healthy when they actually have cancer).

Let's visualize the results!

In [ ]:
# Store metrics
models = ['Decision Tree', 'SVM', 'KNN', 'Naive Bayes']
accuracies = [
    accuracy_score(y_test, y_pred_dt),
    accuracy_score(y_test, y_pred_svm),
    accuracy_score(y_test, y_pred_knn),
    accuracy_score(y_test, y_pred_nb)
]

recalls = [
    recall_score(y_test, y_pred_dt),
    recall_score(y_test, y_pred_svm),
    recall_score(y_test, y_pred_knn),
    recall_score(y_test, y_pred_nb)
]

# Set up the plot
x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, accuracies, width, label='Accuracy', color='#3498db')
rects2 = ax.bar(x + width/2, recalls, width, label='Recall', color='#e74c3c')

# Labels and styling
ax.set_ylabel('Scores', fontsize=12)
ax.set_title('Classification Showdown: Breast Cancer Detection', fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=12)
ax.legend(loc='lower right')
ax.set_ylim([0.8, 1.05]) # Zoom in to see the differences!

# Add values on top of bars
ax.bar_label(rects1, padding=3, fmt='%.3f')
ax.bar_label(rects2, padding=3, fmt='%.3f')

plt.tight_layout()
plt.show()


### Confusion Matrices Extravaganza
Let's see exactly *where* the models went wrong.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Confusion Matrices (Who Misdiagnosed What?)', fontsize=16, fontweight='bold')

preds = [y_pred_dt, y_pred_svm, y_pred_knn, y_pred_nb]
axes = axes.flatten()

for i, (name, pred) in enumerate(zip(models, preds)):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], cbar=False, 
                xticklabels=['Malignant', 'Benign'], 
                yticklabels=['Malignant', 'Benign'],
                annot_kws={"size": 14})
    axes[i].set_title(name, fontsize=14)
    axes[i].set_ylabel('Actual', fontsize=12)
    axes[i].set_xlabel('Predicted', fontsize=12)

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()
